# Tugas 3 — Audit & Pembersihan Korpus Aduan

Visualisasi Data Lanjut (CAK7AGB4) · NIM 203012420009

Input `../datasets/203012420009_aduan.csv`. Output `../datasets/aduan_bersih.csv` (dipakai lagi pekan 10–11).

Urutan pemeriksaan di notebook ini:
1. format tanggal
2. sisa penyandian (entitas/tag HTML)
3. data pribadi → disamarkan
4. normalisasi huruf, tanda baca, spasi
5. kiriman bukan aduan
6. kiriman berulang (setelah normalisasi)
7. kata dipanjangkan & ejaan tak baku
8. campur bahasa & kata henti
9. pemenggalan kata
10. kategori_dilaporkan vs isi teks

Sel bertanda **TODO** berisi keputusan yang perlu diisi/ditinjau sendiri.

In [1]:
import re
import html
from pathlib import Path
from collections import Counter

import pandas as pd

pd.set_option("display.max_colwidth", 200)

DATA = Path("../datasets")
raw = pd.read_csv(DATA / "203012420009_aduan.csv", dtype=str, keep_default_na=False)
N = len(raw)

TEMUAN = []  # baris tabel audit (Lampiran)

def catat(cakupan, jenis, bukti, n, tindakan):
    # bukti & tindakan dari kode; alasan ditulis sendiri di laporan
    TEMUAN.append({"Cakupan": cakupan, "Jenis masalah": jenis, "Bukti": bukti,
                   "Dampak": f"{n} baris ({n / N:.1%})",
                   "Keputusan dan alasan": f"Tindakan: {tindakan}. Alasan: TODO"})

def contoh_frag(pola, teks, k=2):
    # potongan teks paling sering yang cocok dengan pola (bukan kalimat utuh)
    c = Counter(m.group().strip() for s in teks for m in re.finditer(pola, s))
    return "; ".join(f"'{f}' ({n}x)" for f, n in c.most_common(k))
    return n

N, raw.columns.tolist()

(5010,
 ['id_aduan',
  'tanggal',
  'kanal',
  'kelurahan',
  'kategori_dilaporkan',
  'isi_aduan',
  'lon',
  'lat',
  'status'])

## 0. Profil kolom

In [2]:
for c in ["kanal", "kelurahan", "kategori_dilaporkan", "status"]:
    print(raw[c].value_counts().to_string(), "\n")
print("kosong per kolom:")
print((raw.apply(lambda s: s.str.strip()) == "").sum().to_string())
print("id_aduan unik:", raw["id_aduan"].nunique())

kanal
Web                2144
Aplikasi Mobile    1228
WhatsApp            792
Telepon             432
Datang Langsung     285
Surat               129 

kelurahan
Babakan Sari       838
Kebon Gedang       640
Mekarjaya          538
Sekejati           493
Cibuntu Kaler      465
Rancabolang        457
Antapani Wetan     370
Cikawao            346
Cisaranten         277
Margahayu Kidul    246
Ciburial           207
Pasir Wangi        133 

kategori_dilaporkan
Jalan Rusak            915
Penerangan Jalan       818
Sampah                 788
Drainase dan Banjir    576
Fasilitas Umum         537
Ketertiban Umum        508
Air Bersih             464
Lain-lain              404 

status
Selesai             2926
Diteruskan          1241
Ditutup Otomatis     703
Baru                 140 

kosong per kolom:
id_aduan                0
tanggal                 0
kanal                   0
kelurahan               0
kategori_dilaporkan     0
isi_aduan               5
lon                    55
lat          

In [3]:
n_koord = (raw["lon"].str.strip() == "").sum()
catat("lon, lat", "Koordinat kosong", f"{n_koord} baris lon dan lat kosong", n_koord,
      "dibiarkan, baru dipakai pada tugas pekan 10")

## 1. Format tanggal

In [4]:
iso = raw["tanggal"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
miring = raw["tanggal"].str.fullmatch(r"\d{2}/\d{2}/\d{4}")
bag = raw.loc[miring, "tanggal"].str.split("/", expand=True).astype(int)
print("ISO:", iso.sum(), "| xx/xx/yyyy:", miring.sum(), "| lain:", (~iso & ~miring).sum())
# bagian ke-1 > 12 berarti urutannya hari/bulan
print("bagian-1 > 12:", (bag[0] > 12).sum(), "| bagian-2 > 12:", (bag[1] > 12).sum())

tgl = pd.to_datetime(raw["tanggal"].where(iso), format="%Y-%m-%d")
tgl = tgl.fillna(pd.to_datetime(raw["tanggal"].where(miring), format="%d/%m/%Y"))
print("gagal parse:", tgl.isna().sum(), "| rentang:", tgl.min().date(), "s.d.", tgl.max().date())
print(tgl.dt.to_period("M").value_counts().sort_index().to_string())

ISO: 4609 | xx/xx/yyyy: 401 | lain: 0
bagian-1 > 12: 233 | bagian-2 > 12: 0
gagal parse: 0 | rentang: 2025-09-01 s.d. 2026-08-31
tanggal
2025-09    402
2025-10    431
2025-11    427
2025-12    406
2026-01    435
2026-02    390
2026-03    397
2026-04    427
2026-05    453
2026-06    425
2026-07    432
2026-08    385
Freq: M


In [5]:
catat("tanggal", "Format tanggal",
      f"{miring.sum()} baris berformat dd/mm/yyyy, sisanya yyyy-mm-dd; "
      f"{(bag[0] > 12).sum()} baris bagian pertamanya > 12 dan tidak ada yang bagian keduanya > 12 "
      f"(contoh '{raw.loc[miring & (bag.reindex(raw.index)[0] > 12), 'tanggal'].iloc[0]}')", int(miring.sum()),
      "diubah ke datetime dengan dua format eksplisit (%Y-%m-%d dan %d/%m/%Y)")

## 2. Sisa penyandian formulir web

In [6]:
POLA_SANDI = {
    "entitas HTML": r"&(?:[a-zA-Z]+|#\d+);",
    "tag HTML": r"<[^>]+>",
    "persen-encoding": r"%[0-9A-Fa-f]{2}",
    "mojibake": r"Ã.|â€",
    "nbsp / zero-width": "[ ​]",
}
t = raw["isi_aduan"]
for k, p in POLA_SANDI.items():
    print(f"{k:18s}", t.str.contains(p, regex=True).sum())
print(t.str.findall("|".join(POLA_SANDI.values())).explode().value_counts().head(20))

entitas HTML       26
tag HTML           114
persen-encoding    0
mojibake           0
nbsp / zero-width  0
isi_aduan
<br>     267
&amp;     26
Name: count, dtype: int64


In [7]:
sandi = t.str.contains("|".join(POLA_SANDI.values()), regex=True)
n_ent = t.str.contains(POLA_SANDI["entitas HTML"], regex=True).sum()
n_tag = t.str.contains(POLA_SANDI["tag HTML"], regex=True).sum()
catat("isi_aduan", "Sisa penyandian",
      f"{n_ent} baris entitas HTML, {n_tag} baris tag HTML; contoh "
      + contoh_frag("|".join(POLA_SANDI.values()), t), int(sandi.sum()),
      "entitas dikembalikan (html.unescape), tag diganti spasi, sebelum normalisasi")

def bersihkan_sandi(s):
    s = html.unescape(s)
    s = re.sub(r"<[^>]+>", " ", s)
    return s.replace(" ", " ").replace("​", "")

df = raw.assign(tanggal=tgl)
df["teks"] = df["isi_aduan"].map(bersihkan_sandi)
print("sisa setelah dibersihkan:", df["teks"].str.contains("|".join(POLA_SANDI.values()), regex=True).sum())

sisa setelah dibersihkan: 0


## 3. Data pribadi

Pola di bawah perlu ditinjau: cek contoh hasil tangkapan, tambah pola bila ada jenis lain (alamat rumah, plat nomor, dll.).

In [8]:
# kandidat nama orang: sapaan + kata berhuruf kapital (tanpa menyeberangi titik:
# "Bapak/Ibu. Hatur nuhun" adalah awal kalimat baru, bukan nama)
SAPAAN = r"\b(?:Pak|Bu|Bapak|Ibu|Bapa|Pa)\s+([A-Z][a-z]+)"
BUKAN_NAMA = {"Wali", "Mohon", "Atas", "Demikian", "Kami", "Mangga", "Kumaha"}  # jabatan / awal kalimat
kandidat = Counter(m.group(1) for s in df["teks"] for m in re.finditer(SAPAAN, s))
print("kata setelah sapaan:", kandidat.most_common())
sisa = {w: n for w, n in kandidat.items() if re.sub(r"(\w)\1{2,}", r"\1", w) not in BUKAN_NAMA}
print("kandidat nama setelah pengecualian:", {w[:1] + "*" * (len(w) - 1): n for w, n in sisa.items()})

kata setelah sapaan: [('Wali', 520), ('Enok', 107), ('Waliiii', 1), ('Waliii', 1)]
kandidat nama setelah pengecualian: {'E***': 107}


In [9]:
# yang disamarkan hanya grup (?P<pii>...)
LEWATI = "|".join(w[0] + "+".join(w[1:]) + "+" for w in BUKAN_NAMA)  # termasuk bentuk dipanjangkan
POLA_PII = {
    "[EMAIL]": r"(?P<pii>[\w.+-]+@[\w-]+\.[\w.]+)",
    "[NIK]": r"(?<!\d)(?P<pii>\d{16})(?!\d)",
    "[TELP]": r"(?P<pii>(?:\+62|62|0)\s?8[\d\s.-]{7,14}\d)",
    "[NAMA]": r"\b(?:Pak|Bu|Bapak|Ibu|Bapa|Pa)\s+(?P<pii>(?!(?:" + LEWATI + r")\b)[A-Z][a-z]+)",
}
# nama yang ditemukan lewat sapaan dicari lagi tanpa memandang huruf besar-kecil (mis. teks berhuruf kapital semua)
NAMA_DIKENAL = sorted(sisa)
if NAMA_DIKENAL:
    POLA_PII["[NAMA]"] = r"\b(?P<pii>(?i:" + "|".join(NAMA_DIKENAL) + r"))\b"

def cocok(p):
    return df["teks"].map(lambda s: re.search(p, s) is not None)

hit = pd.DataFrame({k: cocok(p) for k, p in POLA_PII.items()})
print(hit.sum().to_string())
print("baris dengan ≥1 data pribadi:", hit.any(axis=1).sum())

[EMAIL]      0
[NIK]       47
[TELP]      98
[NAMA]     115
baris dengan ≥1 data pribadi: 257


In [10]:
# contoh tangkapan per pola; hanya 3 karakter awal yang ditampilkan agar output notebook tetap aman
for k, p in POLA_PII.items():
    print(k, [m.group("pii")[:3] + "*" * (len(m.group("pii")) - 3)
              for s in df["teks"] for m in re.finditer(p, s)][:5])

[EMAIL] []
[NIK] ['327*************', '327*************', '327*************', '327*************', '327*************']
[TELP] ['086*********', '085*********', '087*********', '085*********', '087*********']
[NAMA] ['Eno*', 'Eno*', 'Eno*', 'Eno*', 'Eno*']


In [11]:
def samarkan(s):
    for tanda, p in POLA_PII.items():
        s = re.sub(p, lambda m: m.string[m.start():m.start("pii")] + tanda
                   + m.string[m.end("pii"):m.end()], s)
    return s

df["teks"] = df["teks"].map(samarkan)
print("sisa setelah disamarkan:", sum(cocok(p).sum() for p in POLA_PII.values()))
for k in POLA_PII:
    if hit[k].sum():
        catat("isi_aduan", f"Data pribadi {k}",
              f"{hit[k].sum()} baris; contoh tersamar "
              + contoh_frag(r"(?:\S+\s){0,2}" + re.escape(k) + r"(?:\s\S+){0,1}", df["teks"], 1),
              int(hit[k].sum()), f"diganti tanda {k} sebelum langkah lain")

sisa setelah disamarkan: 0


## 4. Normalisasi huruf, tanda baca, spasi

In [12]:
huruf = df["teks"].str.replace(r"[^A-Za-z]", "", regex=True)
kapital = (huruf.str.len() > 10) & (huruf == huruf.str.upper())
spasi = df["teks"].str.contains(r"\s{2,}|^\s|\s$", regex=True)
print("seluruhnya kapital:", kapital.sum(), "| spasi ganda/di ujung:", spasi.sum())
catat("isi_aduan", "Huruf tak seragam", f"{kapital.sum()} baris seluruhnya huruf kapital", int(kapital.sum()),
      "seluruh teks diturunkan ke huruf kecil")
catat("isi_aduan", "Spasi ganda", f"{spasi.sum()} baris berspasi ganda atau berspasi di ujung", int(spasi.sum()),
      "spasi beruntun disatukan, spasi ujung dibuang")

def normal(s):
    s = s.lower()
    s = re.sub(r"[^\w\s\[\]]", " ", s)   # tanda baca dibuang, token [telp] dll. dipertahankan
    s = re.sub(r"_", " ", s)
    return re.sub(r"\s+", " ", s).strip()

df["norm"] = df["teks"].map(normal)

seluruhnya kapital: 291 | spasi ganda/di ujung: 7


## 5. Kiriman bukan aduan

In [13]:
# teks pendek: tinjau daftarnya untuk menentukan mana yang bukan aduan
pendek = df.loc[df["norm"].str.len() < 25, "norm"]
pendek.value_counts()

norm
              19
coba           8
test           8
tes aja        7
halo           7
asdf           7
test kirim     6
123            6
tes            5
aaa            2
Name: count, dtype: int64

In [14]:
POLA_UJI = r"^(?:tes|test|testing|coba|uji|uji coba|cek|asdf\w*|qwerty|lorem\b.*|halo|p|x+)(?:\s\w+){0,2}$"
kosong = df["norm"] == ""
remeh = ~kosong & (df["norm"].str.replace(" ", "").str.len() <= 3)
uji = ~kosong & ~remeh & df["norm"].str.fullmatch(POLA_UJI)
print("kosong:", kosong.sum(), "| 1-3 karakter:", remeh.sum(), "| teks uji:", uji.sum())
vc = lambda m: "; ".join(f"'{a}' ({b}x)" for a, b in df.loc[m, "norm"].value_counts().head(3).items())
catat("isi_aduan", "Bukan aduan: kosong",
      f"{(raw['isi_aduan'].str.strip() == '').sum()} kosong sejak awal, sisanya kosong setelah tag/tanda baca dibuang",
      int(kosong.sum()), "dibuang")
catat("isi_aduan", "Bukan aduan: 1-3 karakter", vc(remeh), int(remeh.sum()), "dibuang")
catat("isi_aduan", "Bukan aduan: teks uji", vc(uji), int(uji.sum()), "dibuang")
bukan = kosong | remeh | uji
df = df[~bukan].copy()
len(df)

kosong: 19 | 1-3 karakter: 13 | teks uji: 43


4935

## 6. Kiriman berulang

In [15]:
dup_mentah = df.duplicated("isi_aduan").sum()
dup_norm = df.duplicated("norm").sum()
print("duplikat sebelum normalisasi:", dup_mentah, "| sesudah normalisasi:", dup_norm,
      "| hanya tertangkap setelah normalisasi:", dup_norm - dup_mentah)

grup = (df.groupby("norm")
          .agg(n=("id_aduan", "size"), kanal=("kanal", "nunique"),
               varian=("isi_aduan", "nunique"), awal=("tanggal", "min"), akhir=("tanggal", "max"))
          .query("n > 1").sort_values("n", ascending=False))
print("kelompok berulang:", len(grup))
grup.reset_index(drop=True).head(15)

duplikat sebelum normalisasi: 134 | sesudah normalisasi: 150 | hanya tertangkap setelah normalisasi: 16
kelompok berulang: 59


,n,kanal,varian,awal,akhir
0,91,5,11,2025-09-03,2026-08-30
1,3,2,1,2025-09-06,2026-05-25
2,3,2,1,2026-02-26,2026-07-11
3,2,2,2,2025-11-29,2025-12-12
4,2,2,1,2026-01-27,2026-06-30
5,2,2,1,2025-11-19,2026-03-18
6,2,2,2,2026-01-23,2026-06-03
7,2,1,1,2025-11-27,2026-02-08
8,2,1,1,2025-11-30,2026-05-22
9,2,2,1,2025-09-30,2026-06-12


In [16]:
catat("isi_aduan", "Kiriman berulang (identik)",
      f"{len(grup)} kelompok berulang; terbesar {grup['n'].iloc[0]} kiriman lewat {grup['kanal'].iloc[0]} kanal",
      int(dup_mentah), "disatukan, kiriman pertama dipertahankan, jumlah kiriman disimpan di kolom n_kirim")
catat("isi_aduan", "Kiriman berulang (beda tanda baca/huruf)",
      f"{dup_norm - dup_mentah} baris baru tertangkap setelah huruf kecil + tanda baca dibuang "
      f"({(grup['varian'] > 1).sum()} kelompok punya >1 varian tulisan)",
      int(dup_norm - dup_mentah), "sama dengan baris di atas; deduplikasi dilakukan setelah normalisasi")

# disatukan; jumlah kiriman disimpan di n_kirim
df["n_kirim"] = df.groupby("norm")["id_aduan"].transform("size")
df = df.sort_values("tanggal").drop_duplicates("norm", keep="first").copy()
len(df)

4785

## 7. Kata dipanjangkan & ejaan tak baku

In [17]:
POLA_PANJANG = r"([a-z])\1{2,}"
panjang = df["norm"].str.contains(POLA_PANJANG, regex=True)
contoh_panjang = Counter(w for s in df["norm"] for w in s.split() if re.search(POLA_PANJANG, w))
print("baris dengan kata dipanjangkan:", panjang.sum())
print(contoh_panjang.most_common(20))
catat("isi_aduan", "Kata dipanjangkan",
      "contoh " + ", ".join(f"'{w}' ({n}x)" for w, n in contoh_panjang.most_common(3)), int(panjang.sum()),
      "3+ huruf sama diringkas ke bentuk 1 atau 2 huruf yang ada di kosakata korpus")

baris dengan kata dipanjangkan: 185
[('jalannn', 5), ('terimaaaa', 4), ('menggangguuuu', 4), ('baaanyak', 3), ('mohoooon', 3), ('ibuuuuu', 3), ('mohonnn', 3), ('assalamualaikummmm', 3), ('ditindaklanjutiii', 3), ('dalaaam', 3), ('sampaiiiii', 2), ('kamiii', 2), ('teeeerima', 2), ('mooohon', 2), ('teriiiima', 2), ('bantuannyaaa', 2), ('banyaaak', 2), ('dipeeerbaiki', 2), ('atassss', 2), ('gaaaang', 2)]


/tmp/user/1005/ipykernel_3339018/944731480.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  panjang = df["norm"].str.contains(POLA_PANJANG, regex=True)


In [18]:
# 3+ huruf sama → bentuk yang ada di kosakata korpus: 1 huruf (jalannn → jalan) didahulukan,
# lalu 2 huruf (maaaf → maaf); bila keduanya tidak ada → 1 huruf
KOSAKATA_WAJAR = {w for s in df["norm"] for w in s.split() if not re.search(POLA_PANJANG, w)}

def pendek_kata(w):
    if not re.search(POLA_PANJANG, w):
        return w
    satu, dua = re.sub(POLA_PANJANG, r"\1", w), re.sub(POLA_PANJANG, r"\1\1", w)
    if satu in KOSAKATA_WAJAR:
        return satu
    return dua if dua in KOSAKATA_WAJAR else satu

def pendekkan(s):
    return " ".join(pendek_kata(w) for w in s.split())

print(sorted({(w, pendekkan(w)) for w in contoh_panjang})[:40])

[('aaaal', 'al'), ('aaasri', 'asri'), ('aduaaan', 'aduan'), ('anaaak', 'anak'), ('apabilaaa', 'apabila'), ('asihhh', 'asih'), ('asriiii', 'asri'), ('assalamualaikummmm', 'assalamualaikum'), ('atassss', 'atas'), ('baaaantuannya', 'bantuannya'), ('baaaapak', 'bapak'), ('baaanyak', 'banyak'), ('badaaaannnn', 'badan'), ('badaaan', 'badan'), ('badannnn', 'badan'), ('bantuannn', 'bantuan'), ('bantuannyaaa', 'bantuannya'), ('bantuuuannya', 'bantuannya'), ('banyaaak', 'banyak'), ('bapaaaak', 'bapak'), ('beeengkel', 'bengkel'), ('bertuuugas', 'bertugas'), ('bisaaaaa', 'bisa'), ('bisiiiing', 'bising'), ('bisinggggg', 'bising'), ('bukittttt', 'bukit'), ('bumiiii', 'bumi'), ('cendanaaa', 'cendana'), ('daaaalam', 'dalam'), ('daaaapat', 'dapat'), ('daaatang', 'datang'), ('dalaaam', 'dalam'), ('dalammmmm', 'dalam'), ('dapattttt', 'dapat'), ('dariiii', 'dari'), ('deeepan', 'depan'), ('dekatttt', 'dekat'), ('diiiiperbaiki', 'diperbaiki'), ('dilapoooorkan', 'dilaporkan'), ('dipeeerbaiki', 'diperbaiki')]

In [19]:
# kandidat kata tak baku: token pendek yang sering muncul, untuk melengkapi KAMUS_BAKU
frek = Counter(w for s in df["norm"].map(pendekkan) for w in s.split())
[(w, n) for w, n in frek.most_common(400) if len(w) <= 4][:80]

[('di', 5592),
 ('ibu', 4014),
 ('saya', 1832),
 ('kami', 1207),
 ('ke', 1096),
 ('atas', 1039),
 ('dan', 977),
 ('yth', 909),
 ('ini', 891),
 ('gang', 854),
 ('air', 843),
 ('pagi', 705),
 ('yang', 703),
 ('maaf', 701),
 ('hari', 648),
 ('ya', 623),
 ('pak', 623),
 ('bila', 617),
 ('ada', 616),
 ('rw', 524),
 ('mati', 482),
 ('wali', 464),
 ('kota', 464),
 ('bapa', 417),
 ('atuh', 355),
 ('pa', 355),
 ('anak', 319),
 ('raya', 310),
 ('asih', 294),
 ('asri', 286),
 ('kopo', 278),
 ('bumi', 273),
 ('naon', 266),
 ('bau', 244),
 ('dua', 242),
 ('kali', 229),
 ('pada', 225),
 ('lama', 206),
 ('kana', 206),
 ('dari', 205),
 ('bak', 202),
 ('3', 188),
 ('bisa', 186),
 ('pju', 158),
 ('tps', 156),
 ('umum', 142),
 ('sama', 141),
 ('ka', 139),
 ('lami', 137),
 ('07', 135),
 ('teu', 133),
 ('acan', 133),
 ('aya', 133),
 ('saja', 129),
 ('pdam', 126),
 ('al', 124),
 ('04', 121),
 ('liar', 118),
 ('bus', 117),
 ('sd', 111),
 ('bu', 111),
 ('kaki', 106),
 ('lima', 106),
 ('5', 100),
 ('7', 92),
 

In [20]:
# kamus umum + varian yang ditemukan di korpus (sapaan, singkatan surat)
KAMUS_BAKU = {
    "bapa": "bapak", "pa": "bapak", "pak": "bapak", "ngaganggu": "mengganggu", "bu": "ibu", "iibu": "ibu", "yth": "yang terhormat",
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "ngga": "tidak", "enggak": "tidak", "tdk": "tidak",
    "gk": "tidak", "yg": "yang", "dgn": "dengan", "sdh": "sudah", "udah": "sudah", "udh": "sudah",
    "blm": "belum", "blum": "belum", "krn": "karena", "karna": "karena", "utk": "untuk", "bgt": "banget",
    "jg": "juga", "aja": "saja", "dr": "dari", "tp": "tapi", "sy": "saya", "kalo": "kalau", "jln": "jalan",
    "skrg": "sekarang", "sampe": "sampai", "gimana": "bagaimana", "maf": "maaf",
}
tak_baku = df["norm"].map(pendekkan).str.split().map(lambda ws: any(w in KAMUS_BAKU for w in ws))
print("baris dengan kata tak baku (menurut kamus):", tak_baku.sum())
kb = Counter(w for ws in df["norm"].map(pendekkan).str.split() for w in ws if w in KAMUS_BAKU)
catat("isi_aduan", "Ejaan tak baku",
      "contoh " + ", ".join(f"'{w}'→'{KAMUS_BAKU[w]}' ({n}x)" for w, n in kb.most_common(3)), int(tak_baku.sum()),
      f"dibakukan dengan kamus {len(KAMUS_BAKU)} entri")

def bakukan(s):
    return " ".join(KAMUS_BAKU.get(w, w) for w in pendekkan(s).split())

df["norm"] = df["norm"].map(bakukan)

# pemanjangan 2 huruf (teriima, jalaan): diganti bentuk 1 huruf bila bentuk itu jauh lebih sering di korpus
frek_kata = Counter(w for s in df["norm"] for w in s.split())
def pendek_dua(w):
    satu = re.sub(r"([a-z])\1", r"\1", w)
    return satu if satu != w and frek_kata[satu] >= 10 * frek_kata[w] else w
ganti_dua = {w: pendek_dua(w) for w in frek_kata if pendek_dua(w) != w}
print(len(ganti_dua), "kata dipanjangkan 2 huruf:", list(ganti_dua.items())[:15])
df["norm"] = df["norm"].map(lambda s: " ".join(ganti_dua.get(w, w) for w in s.split()))

baris dengan kata tak baku (menurut kamus): 2165


54 kata dipanjangkan 2 huruf: [('jalaan', 'jalan'), ('meeninjau', 'meninjau'), ('moohon', 'mohon'), ('bertugaas', 'bertugas'), ('tindakaan', 'tindakan'), ('enjiing', 'enjing'), ('darii', 'dari'), ('saangat', 'sangat'), ('meembuat', 'membuat'), ('ditindaklaanjuti', 'ditindaklanjuti'), ('teriima', 'terima'), ('ataas', 'atas'), ('ruusak', 'rusak'), ('depaan', 'depan'), ('gaang', 'gang')]


## 8. Campur bahasa & kata henti

In [21]:
# penanda bahasa Sunda
PENANDA_SUNDA = {"teu", "henteu", "abdi", "pisan", "mah", "atuh", "euy", "kumaha", "jeung", "sareng",
                 "punten", "nuhun", "tos", "parantos", "ieu", "eta", "teh", "wae", "oge", "kitu",
                 "geus", "nya", "aya", "anu", "sanes", "acan", "pami", "upami", "bade"}
tok = df["norm"].str.split()
sunda = tok.map(lambda ws: any(w in PENANDA_SUNDA for w in ws))
print("baris memuat penanda Sunda:", sunda.sum())
print(Counter(w for ws in tok for w in ws if w in PENANDA_SUNDA).most_common())
catat("isi_aduan", "Campur bahasa",
      "penanda Sunda terbanyak " + ", ".join(f"'{w}' ({n}x)" for w, n in
      Counter(w for ws in tok for w in ws if w in PENANDA_SUNDA).most_common(3)), int(sunda.sum()),
      "penanda Sunda dimasukkan ke kata henti; pemenggalan kata tidak dipakai")

baris memuat penanda Sunda: 736
[('nuhun', 676), ('punten', 458), ('kumaha', 355), ('atuh', 355), ('parantos', 137), ('teu', 133), ('acan', 133), ('aya', 133)]


In [22]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

STOP_BAWAAN = set(StopWordRemoverFactory().get_stop_words())
print("jumlah kata henti bawaan:", len(STOP_BAWAAN))
# kata ingkar ikut terbuang? ("lampu tidak menyala" → "lampu menyala")
print({w: w in STOP_BAWAAN for w in ["tidak", "belum", "bukan", "tanpa", "jangan", "masih", "sudah", "lagi"]})

jumlah kata henti bawaan: 123
{'tidak': True, 'belum': True, 'bukan': False, 'tanpa': True, 'jangan': False, 'masih': True, 'sudah': True, 'lagi': True}


In [23]:
# kata terbanyak setelah kata henti bawaan dibuang → bahan keputusan kata henti tambahan
frek = Counter(w for ws in tok for w in ws if w not in STOP_BAWAAN)
frek.most_common(60)

[('bapak', 4994),
 ('mohon', 4628),
 ('ibu', 4126),
 ('terima', 4002),
 ('kasih', 4002),
 ('jalan', 3852),
 ('segera', 1784),
 ('warga', 1669),
 ('lokasi', 1647),
 ('ditindaklanjuti', 1469),
 ('mengganggu', 1366),
 ('perhatian', 1340),
 ('depan', 1151),
 ('petugas', 1069),
 ('atas', 1041),
 ('sebelumnya', 930),
 ('terhormat', 909),
 ('banyak', 892),
 ('selamat', 856),
 ('gang', 855),
 ('diperbaiki', 845),
 ('air', 843),
 ('sampah', 811),
 ('lampu', 787),
 ('pagi', 705),
 ('maaf', 701),
 ('perumahan', 677),
 ('hatur', 676),
 ('nuhun', 676),
 ('hari', 649),
 ('bantuannya', 623),
 ('aduan', 620),
 ('bila', 617),
 ('ucapkan', 611),
 ('komplek', 570),
 ('badan', 525),
 ('rw', 524),
 ('sejak', 487),
 ('mati', 482),
 ('bantuan', 466),
 ('kota', 465),
 ('wali', 464),
 ('bertugas', 460),
 ('punten', 458),
 ('datang', 453),
 ('assalamualaikum', 452),
 ('dinas', 445),
 ('terkait', 445),
 ('rusak', 442),
 ('trotoar', 430),
 ('tiang', 424),
 ('siang', 424),
 ('berikut', 424),
 ('perhatiannya', 414)

In [24]:
# kata henti tambahan + alasan (dicantumkan di laporan)
ALASAN = {
    "salam & sopan santun: template surat, ada di hampir semua aduan":
        "mohon terima kasih maaf perhatian perhatiannya atas sebelumnya banyak selamat pagi siang hari "
        "ucapkan bantuan bantuannya assalamualaikum bila",
    "sapaan & penerima: menyebut penerima, bukan masalahnya": "bapak ibu terhormat wali kota dinas terkait",
    "Sunda (template) di luar PENANDA_SUNDA": "hatur mangga",
    "permintaan umum: muncul di semua jenis keluhan, tidak membedakan pokok":
        "segera ditindaklanjuti diperbaiki tindakan aduan berikut menjadi",
    "penanda lokasi: konteks alamat, lokasi dipakai lewat koordinat pada pekan 10":
        "lokasi depan gerbang gang komplek perumahan rw",
    "nama jalan/komplek: penanda lokasi":
        "bukit indah raya kenanga asih griya asri kopo hilir cendana pondok hijau melati bumi lestari "
        "dahlia mawar sukamaju sukaraya",
    "patokan lokasi: tempat rujukan, bukan objek keluhan":
        "seberang dekat belakang pinggir tengah ujung persisnya warung sekolah sd negeri masjid al ikhlas "
        "kantor kelurahan posyandu puskesmas lapangan daerah kawasan rumah",
    "Sunda (template) gelombang kedua":
        "dipiharep enggal waktosna perhatosanana kana wilujeng enjing ditingali heula ka ngantosan lami naon",
    "konteks data pribadi: sisa kalimat 'NIK [NIK] untuk verifikasi', 'nomor ... dihubungi'":
        "nik nomor dihubungi verifikasi",
}
STOP_TAMBAHAN = {w: a for a, ws in ALASAN.items() for w in ws.split()}
# kata ingkar dipertahankan: "air tidak mengalir" ≠ "air mengalir"
TETAP = {"tidak", "belum", "masih"}
STOP = (STOP_BAWAAN | set(STOP_TAMBAHAN) | PENANDA_SUNDA) - TETAP
df["token"] = tok.map(lambda ws: [w for w in ws if w not in STOP and not w.startswith("[") and not w.isdigit()])  # token angka (RT/RW, jam) dibuang
print("rata-rata token per aduan:", df["token"].str.len().mean().round(1))
pd.Series(STOP_TAMBAHAN, name="alasan").groupby(lambda w: STOP_TAMBAHAN[w]).apply(lambda x: ", ".join(x.index))

rata-rata token per aduan: 7.9


Sunda (template) di luar PENANDA_SUNDA                                                                                                                                                                                                                              hatur, mangga
Sunda (template) gelombang kedua                                                                                                                                  dipiharep, enggal, waktosna, perhatosanana, kana, wilujeng, enjing, ditingali, heula, ka, ngantosan, lami, naon
konteks data pribadi: sisa kalimat 'NIK [NIK] untuk verifikasi', 'nomor ... dihubungi'                                                                                                                                                          nik, nomor, dihubungi, verifikasi
nama jalan/komplek: penanda lokasi                                                                                                  bukit, indah, raya, kenanga, asih, griya, asri

## 9. Pemenggalan kata (stemming)

In [25]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
kosakata = Counter(w for ws in df["token"] for w in ws)
akar = {w: stemmer.stem(w) for w in kosakata}
berubah = pd.DataFrame([(w, akar[w], n) for w, n in kosakata.items() if akar[w] != w],
                       columns=["kata", "akar", "frek"]).sort_values("frek", ascending=False)
print("kosakata:", len(kosakata), "| berubah saat dipenggal:", len(berubah))
# cari contoh pemenggalan yang salah di sini
berubah.head(60)

kosakata: 164 | berubah saat dipenggal: 64


,kata,akar,frek
10,mengganggu,ganggu,1366
9,petugas,tugas,1069
16,bertugas,tugas,460
18,berlubang,lubang,410
20,kiranya,kira,364
22,laporan,lapor,351
23,berulang,ulang,351
3,menggenang,genang,340
37,meninjau,tinjau,275
30,melewati,lewat,272


In [26]:
# contoh pemenggalan yang salah (dicantumkan di laporan)
CONTOH_SALAH = ["tumpukan", "berserakan", "lami", "perjalanan", "jalanan", "penerangan", "petugas", "bertugas"]
berubah.set_index("kata").loc[[w for w in CONTOH_SALAH if w in akar and akar[w] != w]]

,akar,frek
kata,,
tumpukan,tumpu,158
berserakan,sera,147
perjalanan,jalan,228
jalanan,jalan,157
penerangan,terang,142
petugas,tugas,1069
bertugas,tugas,460


**Keputusan: tidak memakai pemenggalan kata.** Alasan:

1. **Kosakata kecil.** Setelah kata henti dibuang, kosakata hanya beberapa ratus kata, sehingga pemenggalan hampir tidak mengurangi variasi.
2. **Menyatukan makna yang berbeda.** `perjalanan` dan `jalanan` → `jalan`, padahal Bagian B harus memisahkan keluhan yang memakai kata `jalan`; `petugas`/`bertugas` → `tugas`; `penerangan` → `terang` (istilah "penerangan jalan" hilang).
3. **Menghasilkan akar yang salah.** `tumpukan` → `tumpu` (seharusnya `tumpuk`), `berserakan` → `sera` (seharusnya `serak`).
4. **Merusak kata Sunda.** Stemmer bahasa Indonesia memenggal `lami` (Sunda: lama) → `lam`.

Kolom `token_stem` tetap disimpan untuk pembanding, tetapi analisis Bagian B memakai `token`.

In [27]:
PAKAI_STEM = False
df["token_stem"] = df["token"].map(lambda ws: [akar[w] for w in ws])
print(f"kosakata {len(kosakata)} kata; {len(berubah)} ({len(berubah) / len(kosakata):.0%}) berubah bila dipenggal")

kosakata 164 kata; 64 (39%) berubah bila dipenggal


## 10. kategori_dilaporkan vs isi teks

Kata kunci dipilih dari daftar frekuensi kata; kata yang dipakai lebih dari satu kategori tidak dipakai.
Kategori teks = kategori dengan kata kunci terbanyak; tanpa kecocokan → `tak terdeteksi`.

In [28]:
# bahan kata kunci: seluruh kosakata setelah kata henti, urut frekuensi
" ".join(f"{w}:{n}" for w, n in Counter(w for ws in df["token"] for w in ws).most_common())

'jalan:3852 warga:1669 mengganggu:1366 tidak:1168 petugas:1069 air:843 sampah:811 lampu:787 badan:525 sejak:487 mati:482 bertugas:460 datang:453 rusak:442 trotoar:430 tiang:424 berlubang:410 kiranya:364 listrik:361 apabila:351 laporan:351 berulang:351 waktu:350 menggenang:340 anak:320 minggu:308 taman:294 parkir:280 meninjau:275 langsung:275 melewati:272 menyala:271 kondisi:269 dilaporkan:268 belum:268 dibiarkan:268 khawatir:267 terjadi:267 kecelakaan:267 bau:244 menyengat:244 orang:242 jatuh:242 tersebut:242 besar:242 pohon:236 sekali:233 malam:230 beberapa:229 kali:229 menegur:229 digubris:229 hujan:228 perjalanan:228 total:218 saluran:218 gorong:212 motor:210 lama:206 mengalir:205 bak:202 limbah:192 selama:178 sangat:173 berbahaya:173 penuh:172 meluber:172 berbatu:170 bulan:170 parah:168 lubang:162 membuat:162 pengendara:162 terjatuh:162 pju:158 padam:158 tumpukan:158 menimbulkan:158 jalanan:157 gelap:157 menumpuk:156 tps:156 diangkut:156 aspal:155 mengelupas:155 sepanjang:155 berse

In [29]:
KATA_KUNCI = {
    "Jalan Rusak": ["rusak", "berlubang", "lubang", "aspal", "mengelupas", "berbatu"],
    "Penerangan Jalan": ["lampu", "pju", "padam", "gelap", "redup", "penerangan", "menyala", "listrik"],
    "Sampah": ["sampah", "tps", "diangkut", "menumpuk", "tumpukan", "berserakan"],
    "Drainase dan Banjir": ["drainase", "gorong", "saluran", "tersumbat", "banjir", "menggenang", "meluber",
                            "jebol", "lutut", "setinggi"],
    "Air Bersih": ["pdam", "keruh", "mengalir", "keran", "kaporit", "coklat", "kekuningan", "mengendap",
                   "mandi", "memasak"],
    "Ketertiban Umum": ["parkir", "liar", "pedagang", "kaki", "lima", "bising", "copet", "rawan", "macet",
                        "angkutan", "sembarangan"],
    "Fasilitas Umum": ["halte", "taman", "trotoar"],  # "lapangan" sudah jadi kata henti (patokan)
}
# sengaja tidak dipakai karena dipakai >1 kategori: air, jalan, mati, tiang, kabel, bau

def tebak(ws):
    skor = sorted(((sum(w in v for w in ws), k) for k, v in KATA_KUNCI.items()), reverse=True)
    (n1, k1), (n2, _) = skor[0], skor[1]
    if n1 == 0:
        return "tak terdeteksi"
    return "ambigu" if n1 == n2 else k1  # skor seri → tidak dipaksa ke satu kategori

df["kategori_teks"] = df["token"].map(tebak)
print(df["kategori_teks"].value_counts().to_string(), "\n")
terdeteksi = ~df["kategori_teks"].isin(["tak terdeteksi", "ambigu"])
beda = terdeteksi & (df["kategori_teks"] != df["kategori_dilaporkan"])
print("terdeteksi:", terdeteksi.sum(), "| tidak cocok:", beda.sum(),
      f"({beda.sum() / terdeteksi.sum():.1%} dari yang terdeteksi)")
# "Lain-lain" bukan salah pilih, tetapi warga tidak memilih kategori spesifik
tanpa_lain = terdeteksi & (df["kategori_dilaporkan"] != "Lain-lain")
beda2 = tanpa_lain & (df["kategori_teks"] != df["kategori_dilaporkan"])
print("tanpa Lain-lain → terdeteksi:", tanpa_lain.sum(), "| tidak cocok:", beda2.sum(),
      f"({beda2.sum() / tanpa_lain.sum():.1%})")
print("Lain-lain yang isinya masuk kategori spesifik:",
      (terdeteksi & (df["kategori_dilaporkan"] == "Lain-lain")).sum(), "dari",
      (df["kategori_dilaporkan"] == "Lain-lain").sum())
pd.crosstab(df["kategori_dilaporkan"], df["kategori_teks"], margins=True)

kategori_teks
Jalan Rusak            950
Penerangan Jalan       861
Ketertiban Umum        639
Drainase dan Banjir    624
Sampah                 608
ambigu                 493
tak terdeteksi         291
Air Bersih             272
Fasilitas Umum          47 

terdeteksi: 4001 | tidak cocok: 1273 (31.8% dari yang terdeteksi)
tanpa Lain-lain → terdeteksi: 3724 | tidak cocok: 996 (26.7%)
Lain-lain yang isinya masuk kategori spesifik: 277 dari 381


kategori_teks,Air Bersih,Drainase dan Banjir,Fasilitas Umum,Jalan Rusak,Ketertiban Umum,Penerangan Jalan,Sampah,ambigu,tak terdeteksi,All
kategori_dilaporkan,,,,,,,,,,
Air Bersih,212,21,3,37,13,32,15,42,72,447
Drainase dan Banjir,7,416,0,25,17,33,13,30,12,553
Fasilitas Umum,12,18,20,71,27,130,20,101,111,510
Jalan Rusak,7,19,1,685,21,27,18,92,6,876
Ketertiban Umum,9,24,1,38,347,26,14,18,8,485
Lain-lain,8,81,10,39,87,32,20,45,59,381
Penerangan Jalan,9,28,0,31,108,558,18,20,8,780
Sampah,8,17,12,24,19,23,490,145,15,753
All,272,624,47,950,639,861,608,493,291,4785


In [30]:
catat("kategori_dilaporkan", "Kategori tidak cocok",
      f"{beda.sum()} dari {terdeteksi.sum()} aduan berkategori teks jelas tidak cocok ({beda.sum() / terdeteksi.sum():.1%}); "
      f"tanpa Lain-lain {beda2.sum() / tanpa_lain.sum():.1%}; "
      f"{(terdeteksi & (df['kategori_dilaporkan'] == 'Lain-lain')).sum()} aduan Lain-lain isinya masuk kategori spesifik",
      int(beda.sum()), "TODO keputusan dipakai atau tidak")

## Tabel temuan (Bagian A)

In [31]:
tabel = pd.DataFrame(TEMUAN)
tabel

,Cakupan,Jenis masalah,Bukti,Dampak,Keputusan dan alasan
0,"lon, lat",Koordinat kosong,55 baris lon dan lat kosong,55 baris (1.1%),"Tindakan: dibiarkan, baru dipakai pada tugas pekan 10. Alasan: TODO"
1,tanggal,Format tanggal,"401 baris berformat dd/mm/yyyy, sisanya yyyy-mm-dd; 233 baris bagian pertamanya > 12 dan tidak ada yang bagian keduanya > 12 (contoh '16/12/2025')",401 baris (8.0%),Tindakan: diubah ke datetime dengan dua format eksplisit (%Y-%m-%d dan %d/%m/%Y). Alasan: TODO
2,isi_aduan,Sisa penyandian,"26 baris entitas HTML, 114 baris tag HTML; contoh '<br>' (267x); '&amp;' (26x)",115 baris (2.3%),"Tindakan: entitas dikembalikan (html.unescape), tag diganti spasi, sebelum normalisasi. Alasan: TODO"
3,isi_aduan,Data pribadi [NIK],47 baris; contoh tersamar 'NIK saya [NIK] untuk' (47x),47 baris (0.9%),Tindakan: diganti tanda [NIK] sebelum langkah lain. Alasan: TODO
4,isi_aduan,Data pribadi [TELP],98 baris; contoh tersamar 'Nomor saya [TELP] bisa' (98x),98 baris (2.0%),Tindakan: diganti tanda [TELP] sebelum langkah lain. Alasan: TODO
5,isi_aduan,Data pribadi [NAMA],115 baris; contoh tersamar 'warung Bu [NAMA]' (48x),115 baris (2.3%),Tindakan: diganti tanda [NAMA] sebelum langkah lain. Alasan: TODO
6,isi_aduan,Huruf tak seragam,291 baris seluruhnya huruf kapital,291 baris (5.8%),Tindakan: seluruh teks diturunkan ke huruf kecil. Alasan: TODO
7,isi_aduan,Spasi ganda,7 baris berspasi ganda atau berspasi di ujung,7 baris (0.1%),"Tindakan: spasi beruntun disatukan, spasi ujung dibuang. Alasan: TODO"
8,isi_aduan,Bukan aduan: kosong,"5 kosong sejak awal, sisanya kosong setelah tag/tanda baca dibuang",19 baris (0.4%),Tindakan: dibuang. Alasan: TODO
9,isi_aduan,Bukan aduan: 1-3 karakter,'123' (6x); 'tes' (5x); 'aaa' (2x),13 baris (0.3%),Tindakan: dibuang. Alasan: TODO


## Simpan korpus bersih

In [32]:
keluar = df.assign(token=df["token"].str.join(" "), token_stem=df["token_stem"].str.join(" "))
keluar = keluar[["id_aduan", "tanggal", "kanal", "kelurahan", "kategori_dilaporkan", "kategori_teks",
                 "teks", "norm", "token", "token_stem", "n_kirim", "lon", "lat", "status"]]
keluar = keluar.rename(columns={"teks": "teks_samaran"})
keluar.to_csv(DATA / "aduan_bersih.csv", index=False)
keluar.shape

(4785, 14)